In [2]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from tqdm import tqdm

In [3]:
class RWFDataset(Dataset):
    def __init__(self, root, frames=32):
        self.samples = []
        self.frames = frames

        for label, cls in enumerate(["NonFight", "Fight"]):
            folder = os.path.join(root, cls)
            for file in os.listdir(folder):
                if file.endswith(".avi") or file.endswith(".mp4"):
                    self.samples.append((os.path.join(folder, file), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        cap = cv2.VideoCapture(path)
        frames = []

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.resize(frame, (224,224))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # horizontal flip (augmentation)
            if np.random.rand() < 0.5:
                frame = cv2.flip(frame, 1)

            frames.append(frame)

        cap.release()

        if len(frames) == 0:
            frames = [np.zeros((224,224,3), dtype=np.uint8)]

        if len(frames) >= self.frames:
            idxs = np.linspace(0, len(frames)-1, self.frames).astype(int)
            frames = [frames[i] for i in idxs]
        else:
            frames += [frames[-1]] * (self.frames - len(frames))

        # ImageNet normalization
        mean = np.array([0.485, 0.456, 0.406])
        std  = np.array([0.229, 0.224, 0.225])
        clip = (np.array(frames)/255.0 - mean) / std

        clip = torch.tensor(clip).permute(0,3,1,2).float()
        return clip, torch.tensor(label, dtype=torch.float32)

In [4]:
train_set = RWFDataset(r"D:\Dtaset fot cctv\RWF-2000\train")
val_set   = RWFDataset(r"D:\Dtaset fot cctv\RWF-2000\val")

train_loader = DataLoader(train_set, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_set, batch_size=4)

In [5]:
class Stage2Model(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet18(weights="IMAGENET1K_V1")
        self.cnn = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        B,T,C,H,W = x.shape
        feats = []
        for t in range(T):
            f = self.cnn(x[:,t]).view(B,-1)
            feats.append(f)
        feats = torch.stack(feats,1).mean(1)
        return self.fc(feats)

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = Stage2Model().to(device)
model.load_state_dict(torch.load("stage2_rwf_epoch_8.pth", map_location=device))

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

EPOCHS = 15
start_epoch = 8

In [7]:
for epoch in range(start_epoch, EPOCHS):
    model.train()
    correct, total = 0, 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for clips, labels in pbar:
        clips, labels = clips.to(device), labels.to(device)

        preds = model(clips).view(-1)
        loss = criterion(preds, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        probs = torch.sigmoid(preds)
        preds_bin = (probs > 0.5).float()

        correct += (preds_bin == labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix(acc=correct/total, loss=loss.item())

    train_acc = correct / total

    # -------- Validation --------
    model.eval()
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for clips, labels in val_loader:
            clips, labels = clips.to(device), labels.to(device)

            preds = model(clips).view(-1)
            probs = torch.sigmoid(preds)
            preds_bin = (probs > 0.5).float()

            val_correct += (preds_bin == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    torch.save(model.state_dict(), f"stage2_rwf_epoch_{epoch+1}.pth")

Epoch 9/15: 100%|██████████| 400/400 [1:13:03<00:00, 10.96s/it, acc=0.914, loss=0.206] 


Epoch 9 | Train Acc: 0.9144 | Val Acc: 0.7100


Epoch 10/15:   8%|▊         | 32/400 [05:17<1:00:51,  9.92s/it, acc=0.945, loss=0.0573]


KeyboardInterrupt: 